# Use Case 05: Codebase Documentation Generator

**Companion notebook for**: `05-codebase-documentation.html`

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai/usecases/notebooks/05-codebase-documentation.ipynb)

## What We Build
An LLM-powered pipeline that:
1. Parses Python source code using AST (Abstract Syntax Trees)
2. Extracts function signatures, class hierarchies, and module dependencies
3. Generates accurate docstrings using an LLM with rich context
4. Creates architecture diagrams in Mermaid syntax
5. Calculates documentation coverage metrics
6. Detects documentation drift

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q openai tiktoken

In [ ]:
import os
import ast
import json
import textwrap
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

import tiktoken
from openai import OpenAI

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
client = OpenAI()

print("All imports successful. OpenAI API key found.")

---
## 1. Sample Python Source Code

We define a realistic Python module with a mix of documented and undocumented functions and classes.
This serves as our test codebase.

In [ ]:
SAMPLE_MODULE = '''
"""User management module for the CareerAlign platform.

Handles user creation, authentication, profile updates, and role management.
"""

import hashlib
import hmac
import secrets
import re
from datetime import datetime, timedelta
from typing import Optional
from dataclasses import dataclass, field


# Constants
PASSWORD_MIN_LENGTH = 8
TOKEN_EXPIRY_HOURS = 24
MAX_LOGIN_ATTEMPTS = 5


@dataclass
class UserProfile:
    """Represents a user profile with personal and professional information."""
    user_id: str
    email: str
    full_name: str
    role: str = "member"
    is_active: bool = True
    created_at: datetime = field(default_factory=datetime.utcnow)
    skills: list[str] = field(default_factory=list)
    login_attempts: int = 0

    def is_admin(self) -> bool:
        return self.role == "admin"

    def add_skill(self, skill: str) -> None:
        if skill not in self.skills:
            self.skills.append(skill)

    def remove_skill(self, skill: str) -> bool:
        if skill in self.skills:
            self.skills.remove(skill)
            return True
        return False


class AuthenticationError(Exception):
    """Raised when authentication fails."""
    pass


class RateLimitError(Exception):
    pass


class UserManager:
    def __init__(self, db_connection):
        self._db = db_connection
        self._users: dict[str, UserProfile] = {}
        self._tokens: dict[str, tuple[str, datetime]] = {}

    def create_user(self, email: str, full_name: str, password: str,
                    role: str = "member") -> UserProfile:
        if not self._validate_email(email):
            raise ValueError(f"Invalid email: {email}")
        if len(password) < PASSWORD_MIN_LENGTH:
            raise ValueError(f"Password must be at least {PASSWORD_MIN_LENGTH} characters")
        if email in {u.email for u in self._users.values()}:
            raise ValueError(f"User with email {email} already exists")

        user_id = secrets.token_hex(16)
        password_hash = self._hash_password(password)
        user = UserProfile(
            user_id=user_id,
            email=email,
            full_name=full_name,
            role=role,
        )
        self._users[user_id] = user
        self._db.store(user_id, {"profile": user, "password_hash": password_hash})
        return user

    def authenticate(self, email: str, password: str) -> str:
        user = self._find_by_email(email)
        if user is None:
            raise AuthenticationError("User not found")

        if user.login_attempts >= MAX_LOGIN_ATTEMPTS:
            raise RateLimitError("Account locked due to too many failed attempts")

        stored = self._db.get(user.user_id)
        if not self._verify_password(password, stored["password_hash"]):
            user.login_attempts += 1
            raise AuthenticationError("Invalid password")

        user.login_attempts = 0
        token = self._generate_token(user.user_id)
        return token

    def get_user(self, user_id: str) -> Optional[UserProfile]:
        return self._users.get(user_id)

    def update_profile(self, user_id: str, **kwargs) -> UserProfile:
        user = self._users.get(user_id)
        if user is None:
            raise ValueError(f"User {user_id} not found")
        for key, value in kwargs.items():
            if hasattr(user, key) and key not in ("user_id", "created_at"):
                setattr(user, key, value)
        return user

    def deactivate_user(self, user_id: str) -> bool:
        user = self._users.get(user_id)
        if user:
            user.is_active = False
            return True
        return False

    def list_users(self, role: Optional[str] = None,
                   active_only: bool = True) -> list[UserProfile]:
        users = list(self._users.values())
        if active_only:
            users = [u for u in users if u.is_active]
        if role:
            users = [u for u in users if u.role == role]
        return users

    def validate_token(self, token: str) -> Optional[str]:
        if token not in self._tokens:
            return None
        user_id, expires = self._tokens[token]
        if datetime.utcnow() > expires:
            del self._tokens[token]
            return None
        return user_id

    def _validate_email(self, email: str) -> bool:
        pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$"
        return bool(re.match(pattern, email))

    def _hash_password(self, password: str) -> str:
        salt = secrets.token_hex(16)
        hash_val = hashlib.pbkdf2_hmac("sha256", password.encode(), salt.encode(), 100_000)
        return f"{salt}:{hash_val.hex()}"

    def _verify_password(self, password: str, stored_hash: str) -> bool:
        salt, hash_val = stored_hash.split(":")
        new_hash = hashlib.pbkdf2_hmac("sha256", password.encode(), salt.encode(), 100_000)
        return hmac.compare_digest(new_hash.hex(), hash_val)

    def _find_by_email(self, email: str) -> Optional[UserProfile]:
        for user in self._users.values():
            if user.email == email:
                return user
        return None

    def _generate_token(self, user_id: str) -> str:
        token = secrets.token_urlsafe(32)
        expires = datetime.utcnow() + timedelta(hours=TOKEN_EXPIRY_HOURS)
        self._tokens[token] = (user_id, expires)
        return token


def format_user_summary(user: UserProfile) -> str:
    skills_str = ", ".join(user.skills) if user.skills else "No skills listed"
    return (
        f"{user.full_name} ({user.email})\\n"
        f"Role: {user.role} | Active: {user.is_active}\\n"
        f"Skills: {skills_str}"
    )


def bulk_import_users(user_manager: "UserManager", user_data: list[dict]) -> dict:
    results = {"created": [], "failed": []}
    for data in user_data:
        try:
            user = user_manager.create_user(
                email=data["email"],
                full_name=data["name"],
                password=data.get("password", secrets.token_urlsafe(12)),
                role=data.get("role", "member"),
            )
            results["created"].append(user.user_id)
        except (ValueError, KeyError) as e:
            results["failed"].append({"data": data, "error": str(e)})
    return results
'''

print(f"Sample module: {len(SAMPLE_MODULE.splitlines())} lines")

---
## 2. AST Parsing — Extract All Functions and Classes

We parse the sample module into an AST and extract structured metadata for every function and class.

In [ ]:
@dataclass
class FunctionInfo:
    """Metadata extracted from a single function definition."""
    name: str
    args: list[str]
    arg_annotations: dict[str, str]
    return_type: Optional[str] = None
    docstring: Optional[str] = None
    decorators: list[str] = field(default_factory=list)
    line_number: int = 0
    end_line: int = 0
    is_method: bool = False
    is_public: bool = True
    parent_class: Optional[str] = None
    source: str = ""


@dataclass
class ClassInfo:
    """Metadata extracted from a class definition."""
    name: str
    bases: list[str]
    docstring: Optional[str] = None
    methods: list[FunctionInfo] = field(default_factory=list)
    decorators: list[str] = field(default_factory=list)
    line_number: int = 0
    end_line: int = 0
    source: str = ""


@dataclass
class ModuleInfo:
    """All metadata extracted from a single Python module."""
    module_docstring: Optional[str] = None
    imports: list[str] = field(default_factory=list)
    functions: list[FunctionInfo] = field(default_factory=list)
    classes: list[ClassInfo] = field(default_factory=list)
    constants: list[str] = field(default_factory=list)


def _get_source_segment(source: str, node: ast.AST) -> str:
    """Extract source code for a given AST node."""
    lines = source.splitlines(keepends=True)
    start = node.lineno - 1
    end = node.end_lineno
    return ''.join(lines[start:end])


def parse_module(source: str) -> ModuleInfo:
    """Parse a Python source string and extract all metadata."""
    tree = ast.parse(source)
    info = ModuleInfo()
    info.module_docstring = ast.get_docstring(tree)

    for node in ast.iter_child_nodes(tree):
        # Imports
        if isinstance(node, ast.Import):
            for alias in node.names:
                info.imports.append(alias.name)
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ''
            for alias in node.names:
                info.imports.append(f"{module}.{alias.name}")

        # Constants (top-level uppercase assignments)
        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id.isupper():
                    info.constants.append(target.id)

        # Top-level functions
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            func = _extract_function(node, source)
            info.functions.append(func)

        # Classes
        elif isinstance(node, ast.ClassDef):
            cls = _extract_class(node, source)
            info.classes.append(cls)

    return info


def _extract_function(node, source: str, parent_class: str = None) -> FunctionInfo:
    """Extract metadata from a function/method AST node."""
    arg_annotations = {}
    for arg in node.args.args:
        if arg.annotation:
            arg_annotations[arg.arg] = ast.unparse(arg.annotation)

    return FunctionInfo(
        name=node.name,
        args=[arg.arg for arg in node.args.args],
        arg_annotations=arg_annotations,
        return_type=ast.unparse(node.returns) if node.returns else None,
        docstring=ast.get_docstring(node),
        decorators=[ast.unparse(d) for d in node.decorator_list],
        line_number=node.lineno,
        end_line=node.end_lineno,
        is_method=parent_class is not None,
        is_public=not node.name.startswith('_'),
        parent_class=parent_class,
        source=_get_source_segment(source, node),
    )


def _extract_class(node: ast.ClassDef, source: str) -> ClassInfo:
    """Extract metadata from a class AST node."""
    methods = []
    for item in node.body:
        if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)):
            methods.append(_extract_function(item, source, parent_class=node.name))

    return ClassInfo(
        name=node.name,
        bases=[ast.unparse(b) for b in node.bases],
        docstring=ast.get_docstring(node),
        methods=methods,
        decorators=[ast.unparse(d) for d in node.decorator_list],
        line_number=node.lineno,
        end_line=node.end_lineno,
        source=_get_source_segment(source, node),
    )


# Parse our sample module
module_info = parse_module(SAMPLE_MODULE)

print(f"Module docstring: {module_info.module_docstring[:60]}...")
print(f"Imports: {len(module_info.imports)}")
print(f"Constants: {module_info.constants}")
print(f"Top-level functions: {[f.name for f in module_info.functions]}")
print(f"Classes: {[c.name for c in module_info.classes]}")
for cls in module_info.classes:
    print(f"  {cls.name} (bases: {cls.bases}) — {len(cls.methods)} methods")
    for m in cls.methods:
        has_doc = 'YES' if m.docstring else 'NO'
        print(f"    .{m.name}() — docstring: {has_doc}")

---
## 3. Identify Undocumented Functions

We scan all extracted functions and classes to find those missing meaningful docstrings.

In [ ]:
def find_undocumented(module: ModuleInfo, min_docstring_length: int = 20) -> list[FunctionInfo]:
    """Find all public functions/methods without meaningful docstrings."""
    undocumented = []

    # Check top-level functions
    for func in module.functions:
        if func.is_public and (
            not func.docstring or len(func.docstring.strip()) < min_docstring_length
        ):
            undocumented.append(func)

    # Check class methods
    for cls in module.classes:
        for method in cls.methods:
            if method.is_public and (
                not method.docstring or len(method.docstring.strip()) < min_docstring_length
            ):
                undocumented.append(method)

    return undocumented


undocumented = find_undocumented(module_info)

print(f"Found {len(undocumented)} undocumented public functions/methods:\n")
for func in undocumented:
    location = f"{func.parent_class}.{func.name}" if func.parent_class else func.name
    print(f"  - {location}() at line {func.line_number}")
    print(f"    args: {func.args}")
    print(f"    return: {func.return_type}")
    print()

---
## 4. LLM-Powered Docstring Generation

For each undocumented function, we send its source code (plus module context) to the LLM and get back a properly formatted docstring.

In [ ]:
DOCSTRING_SYSTEM_PROMPT = """You are a senior Python developer writing precise, Google-style docstrings.

Rules:
1. Describe WHAT the function does, not HOW it does it internally.
2. Document every parameter with its type and purpose.
3. Document the return value with its type and meaning.
4. Include a Raises section if the function raises exceptions.
5. Be precise: if a parameter is Optional, say so.
6. Return ONLY the docstring text, no triple quotes, no code fences.
7. Use Google-style format (Args:, Returns:, Raises: sections).
"""


def generate_docstring(func: FunctionInfo, module_context: str = "") -> str:
    """Generate a docstring for a function using the LLM."""
    prompt = f"""Generate a Google-style Python docstring for this function.

FUNCTION SOURCE:
```python
{func.source}
```

MODULE CONTEXT (the module this function belongs to):
{module_context[:2000] if module_context else 'Not provided.'}

PARENT CLASS: {func.parent_class or 'None (top-level function)'}

Return ONLY the docstring text. No triple quotes. No markdown fences."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": DOCSTRING_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=500,
    )
    return response.choices[0].message.content.strip()


# Generate docstrings for the first 3 undocumented functions as a demo
demo_functions = undocumented[:3]
generated_docs = {}

# Build a short module context string
module_context = f"""Module: user_management
Docstring: {module_info.module_docstring}
Constants: {module_info.constants}
Classes: {[c.name for c in module_info.classes]}
Top-level functions: {[f.name for f in module_info.functions]}"""

for func in demo_functions:
    location = f"{func.parent_class}.{func.name}" if func.parent_class else func.name
    print(f"Generating docstring for: {location}()")
    print("-" * 60)

    docstring = generate_docstring(func, module_context)
    generated_docs[location] = docstring

    print(docstring)
    print("\n" + "=" * 60 + "\n")

---
## 5. Module-Level Documentation Generation

Beyond individual docstrings, we generate a comprehensive module-level documentation summary that describes the module's purpose, its classes, key functions, and how they relate.

In [ ]:
def generate_module_docs(module: ModuleInfo) -> str:
    """Generate comprehensive module-level documentation."""

    # Build a structured summary of the module
    classes_summary = []
    for cls in module.classes:
        methods_list = [f"  - {m.name}({', '.join(m.args)})" for m in cls.methods]
        classes_summary.append(
            f"Class: {cls.name}\n"
            f"  Bases: {cls.bases}\n"
            f"  Docstring: {cls.docstring or 'None'}\n"
            f"  Methods:\n" + "\n".join(methods_list)
        )

    functions_summary = []
    for func in module.functions:
        functions_summary.append(
            f"Function: {func.name}({', '.join(func.args)}) -> {func.return_type}\n"
            f"  Docstring: {func.docstring or 'None'}"
        )

    prompt = f"""Generate comprehensive module-level documentation for this Python module.

MODULE DOCSTRING: {module.module_docstring or 'None'}

IMPORTS: {module.imports}

CONSTANTS: {module.constants}

CLASSES:
{chr(10).join(classes_summary)}

TOP-LEVEL FUNCTIONS:
{chr(10).join(functions_summary)}

Generate documentation that includes:
1. Module overview (2-3 sentences)
2. Quick start / usage example
3. Class descriptions with their responsibilities
4. Key function descriptions
5. Important constants and their meanings
6. Dependencies

Format as clean Markdown."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a technical writer producing clean, accurate module documentation."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=1500,
    )
    return response.choices[0].message.content.strip()


module_docs = generate_module_docs(module_info)
print(module_docs)

---
## 6. Mermaid Diagram Generation

We generate Mermaid diagrams to visualize the class hierarchy and module structure. Mermaid renders natively in GitHub, GitLab, Notion, and most documentation platforms.

In [ ]:
def generate_class_diagram(module: ModuleInfo) -> str:
    """Generate a Mermaid class diagram from module metadata."""
    lines = ["classDiagram"]

    for cls in module.classes:
        # Class definition with methods
        lines.append(f"    class {cls.name} {{")
        for method in cls.methods:
            visibility = "+" if method.is_public else "-"
            args_str = ", ".join(
                a for a in method.args if a != "self"
            )
            ret = method.return_type or "None"
            lines.append(f"        {visibility}{method.name}({args_str}) {ret}")
        lines.append("    }")

        # Inheritance relationships
        for base in cls.bases:
            if base not in ("object",):
                lines.append(f"    {base} <|-- {cls.name}")

    return "\n".join(lines)


def generate_module_flowchart(module: ModuleInfo) -> str:
    """Generate a Mermaid flowchart showing key relationships."""
    lines = ["graph TD"]

    # Show classes and their key public methods
    for cls in module.classes:
        cls_id = cls.name
        lines.append(f"    {cls_id}[{cls.name}]")
        for method in cls.methods:
            if method.is_public:
                method_id = f"{cls_id}_{method.name}"
                lines.append(f"    {method_id}[.{method.name}]")
                lines.append(f"    {cls_id} --> {method_id}")

    # Show top-level functions
    for func in module.functions:
        func_id = func.name
        lines.append(f"    {func_id}[{func.name}]")

    # Show relationships between functions and classes they reference
    for func in module.functions:
        for cls in module.classes:
            # Check if function references the class in its type annotations
            for ann in func.arg_annotations.values():
                if cls.name in ann:
                    lines.append(f"    {func.name} -.-> {cls.name}")
                    break

    return "\n".join(lines)


# Generate class diagram
class_diagram = generate_class_diagram(module_info)
print("=== CLASS DIAGRAM ===")
print(class_diagram)

print("\n\n=== MODULE FLOWCHART ===")
flowchart = generate_module_flowchart(module_info)
print(flowchart)

---
## 7. LLM-Enhanced Architecture Diagram

We can also ask the LLM to generate a richer Mermaid diagram that captures data flow and relationships that are hard to detect from AST alone.

In [ ]:
def generate_architecture_diagram(module: ModuleInfo) -> str:
    """Use LLM to generate a rich architecture diagram in Mermaid syntax."""

    # Prepare a summary of the module structure
    summary_parts = []
    for cls in module.classes:
        methods = [m.name for m in cls.methods]
        summary_parts.append(f"Class {cls.name} (bases: {cls.bases}): methods = {methods}")
    for func in module.functions:
        summary_parts.append(f"Function {func.name}({', '.join(func.args)}) -> {func.return_type}")

    prompt = f"""Given this Python module structure, generate a Mermaid diagram that shows
the architecture and data flow.

Module: user_management
Purpose: {module.module_docstring}

Structure:
{chr(10).join(summary_parts)}

Constants: {module.constants}
Imports: {module.imports}

Requirements:
1. Use 'graph TD' (top-down flowchart)
2. Show classes as rounded boxes
3. Show data flow between components
4. Group related items with subgraph
5. Show external dependencies (hashlib, secrets, etc.)
6. Return ONLY valid Mermaid syntax, no markdown fences
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You generate valid Mermaid diagram syntax. Return only the diagram, no explanations."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=800,
    )
    return response.choices[0].message.content.strip()


architecture_diagram = generate_architecture_diagram(module_info)
print("=== LLM-GENERATED ARCHITECTURE DIAGRAM ===")
print(architecture_diagram)

---
## 8. Documentation Coverage Calculation

We calculate what percentage of public functions and classes have meaningful docstrings, producing a coverage report similar to test coverage.

In [ ]:
def calculate_coverage(module: ModuleInfo, min_length: int = 20) -> dict:
    """Calculate documentation coverage metrics.

    Args:
        module: Parsed module metadata.
        min_length: Minimum docstring length to count as "documented".

    Returns:
        Dict with coverage statistics.
    """
    items = []

    # Top-level functions
    for func in module.functions:
        items.append({
            "name": func.name,
            "type": "function",
            "is_public": func.is_public,
            "has_docstring": bool(func.docstring and len(func.docstring.strip()) >= min_length),
            "line": func.line_number,
        })

    # Classes
    for cls in module.classes:
        items.append({
            "name": cls.name,
            "type": "class",
            "is_public": not cls.name.startswith("_"),
            "has_docstring": bool(cls.docstring and len(cls.docstring.strip()) >= min_length),
            "line": cls.line_number,
        })

        # Methods
        for method in cls.methods:
            items.append({
                "name": f"{cls.name}.{method.name}",
                "type": "method",
                "is_public": method.is_public,
                "has_docstring": bool(method.docstring and len(method.docstring.strip()) >= min_length),
                "line": method.line_number,
            })

    # Calculate metrics
    public_items = [i for i in items if i["is_public"]]
    documented = [i for i in public_items if i["has_docstring"]]
    undocumented = [i for i in public_items if not i["has_docstring"]]

    total = len(public_items)
    doc_count = len(documented)

    return {
        "total_public": total,
        "documented": doc_count,
        "undocumented_count": total - doc_count,
        "coverage_pct": round(doc_count / total * 100, 1) if total > 0 else 0,
        "documented_items": [i["name"] for i in documented],
        "undocumented_items": [i["name"] for i in undocumented],
        "all_items": items,
    }


coverage = calculate_coverage(module_info)

print("=" * 50)
print("DOCUMENTATION COVERAGE REPORT")
print("=" * 50)
print(f"\nTotal public items:    {coverage['total_public']}")
print(f"Documented:            {coverage['documented']}")
print(f"Undocumented:          {coverage['undocumented_count']}")
print(f"Coverage:              {coverage['coverage_pct']}%")
print(f"\nDocumented items:")
for name in coverage["documented_items"]:
    print(f"  [OK]  {name}")
print(f"\nUndocumented items:")
for name in coverage["undocumented_items"]:
    print(f"  [!!]  {name}")

# Visual coverage bar
bar_len = 40
filled = int(bar_len * coverage['coverage_pct'] / 100)
bar = '#' * filled + '-' * (bar_len - filled)
print(f"\n  [{bar}] {coverage['coverage_pct']}%")

---
## 9. Full Pipeline — End to End

Let us run the complete documentation pipeline: parse, analyze, generate docstrings for all undocumented functions, generate module docs, generate diagrams, and report coverage.

In [ ]:
def run_documentation_pipeline(source_code: str, module_name: str = "module") -> dict:
    """Run the full documentation generation pipeline.

    Args:
        source_code: Python source code as a string.
        module_name: Name of the module for labeling.

    Returns:
        Dict with all generated documentation artifacts.
    """
    print(f"{'=' * 60}")
    print(f"DOCUMENTATION PIPELINE: {module_name}")
    print(f"{'=' * 60}")

    # Step 1: Parse
    print("\n[1/6] Parsing source code with AST...")
    module = parse_module(source_code)
    print(f"  Found: {len(module.classes)} classes, "
          f"{len(module.functions)} functions, "
          f"{len(module.imports)} imports")

    # Step 2: Calculate initial coverage
    print("\n[2/6] Calculating initial coverage...")
    initial_coverage = calculate_coverage(module)
    print(f"  Initial coverage: {initial_coverage['coverage_pct']}%")
    print(f"  Undocumented: {initial_coverage['undocumented_count']} items")

    # Step 3: Find undocumented items
    print("\n[3/6] Identifying undocumented functions...")
    undoc = find_undocumented(module)
    print(f"  {len(undoc)} functions need docstrings")

    # Step 4: Generate docstrings
    print("\n[4/6] Generating docstrings with LLM...")
    context = f"Module: {module_name}\nDocstring: {module.module_docstring}"
    docstrings = {}
    for func in undoc:
        loc = f"{func.parent_class}.{func.name}" if func.parent_class else func.name
        print(f"  Generating: {loc}()")
        docstrings[loc] = generate_docstring(func, context)
    print(f"  Generated {len(docstrings)} docstrings")

    # Step 5: Generate diagrams
    print("\n[5/6] Generating Mermaid diagrams...")
    class_diag = generate_class_diagram(module)
    flow_diag = generate_module_flowchart(module)
    arch_diag = generate_architecture_diagram(module)
    print("  Generated: class diagram, flowchart, architecture diagram")

    # Step 6: Generate module docs
    print("\n[6/6] Generating module-level documentation...")
    mod_docs = generate_module_docs(module)
    print("  Module documentation generated")

    print(f"\n{'=' * 60}")
    print("PIPELINE COMPLETE")
    print(f"{'=' * 60}")

    return {
        "module_name": module_name,
        "initial_coverage": initial_coverage,
        "generated_docstrings": docstrings,
        "class_diagram": class_diag,
        "flowchart": flow_diag,
        "architecture_diagram": arch_diag,
        "module_docs": mod_docs,
        "module_info": module,
    }


# Run the full pipeline
results = run_documentation_pipeline(SAMPLE_MODULE, "user_management")

In [ ]:
# Display all generated docstrings
print("=" * 60)
print("ALL GENERATED DOCSTRINGS")
print("=" * 60)

for func_name, docstring in results["generated_docstrings"].items():
    print(f"\n--- {func_name}() ---")
    print(docstring)
    print()

In [ ]:
# Display the generated module documentation
print("=" * 60)
print("MODULE DOCUMENTATION")
print("=" * 60)
print()
print(results["module_docs"])

In [ ]:
# Display diagrams
print("=== CLASS DIAGRAM (paste into any Mermaid renderer) ===")
print(results["class_diagram"])
print("\n\n=== ARCHITECTURE DIAGRAM ===")
print(results["architecture_diagram"])

---
## 10. Docstring Validation

We validate that generated docstrings match the actual function signatures to catch hallucinations.

In [ ]:
def validate_docstring(func: FunctionInfo, docstring: str) -> dict:
    """Validate a generated docstring against the function signature.

    Checks:
    - All parameters are mentioned
    - No hallucinated parameters
    - Return type consistency
    """
    issues = []

    # Get actual params (exclude 'self' for methods)
    actual_params = [a for a in func.args if a != "self"]

    # Check each actual param is mentioned in the docstring
    docstring_lower = docstring.lower()
    for param in actual_params:
        if param.lower() not in docstring_lower:
            issues.append(f"MISSING_PARAM: '{param}' not mentioned in docstring")

    # Check return type if function has one
    if func.return_type:
        # Normalize common type representations
        ret_type_lower = func.return_type.lower()
        has_returns_section = "returns:" in docstring_lower or "return" in docstring_lower
        if not has_returns_section and ret_type_lower != "none":
            issues.append(f"MISSING_RETURNS: function returns {func.return_type} but no Returns section")

    # Check for minimum quality
    if len(docstring.strip()) < 20:
        issues.append("TOO_SHORT: docstring is less than 20 characters")

    return {
        "function": f"{func.parent_class}.{func.name}" if func.parent_class else func.name,
        "valid": len(issues) == 0,
        "issues": issues,
        "param_count": len(actual_params),
    }


# Validate all generated docstrings
print("DOCSTRING VALIDATION REPORT")
print("=" * 50)

undoc_funcs = find_undocumented(module_info)
validation_results = []

for func in undoc_funcs:
    loc = f"{func.parent_class}.{func.name}" if func.parent_class else func.name
    if loc in results["generated_docstrings"]:
        result = validate_docstring(func, results["generated_docstrings"][loc])
        validation_results.append(result)
        status = "PASS" if result["valid"] else "FAIL"
        print(f"  [{status}] {result['function']}")
        for issue in result["issues"]:
            print(f"         {issue}")

passed = sum(1 for r in validation_results if r["valid"])
total = len(validation_results)
print(f"\nValidation: {passed}/{total} passed ({round(passed/total*100) if total else 0}%)")

---
## Summary

In this notebook we built a complete **Codebase Documentation Generator** pipeline:

1. **AST Parsing** — Extracted structured metadata from Python source code
2. **Undocumented Detection** — Identified functions and methods lacking docstrings
3. **LLM Docstring Generation** — Generated Google-style docstrings with contextual prompts
4. **Module Documentation** — Created comprehensive module-level docs
5. **Mermaid Diagrams** — Generated class and architecture diagrams
6. **Coverage Metrics** — Calculated documentation coverage percentage
7. **Validation** — Verified generated docstrings against function signatures

### Key Takeaways
- AST parsing is far more reliable than regex for code analysis
- Providing rich context (module structure, tests, types) dramatically improves LLM output quality
- Low temperature (0.2) produces more accurate, deterministic documentation
- Validation catches hallucinated parameters and missing return type descriptions
- The pipeline integrates naturally with CI/CD for continuous documentation maintenance